## Document Loader

In [1]:
from pypdf import PdfReader
import os

def load_pdfs(folder_path):
    docs = []

    for file in os.listdir(folder_path):
        if not file.endswith(".pdf"):
            continue

        path = os.path.join(folder_path, file)
        reader = PdfReader(path)

        for page_num, page in enumerate(reader.pages):
            text = page.extract_text()

            if text and text.strip():
                docs.append({
                    "text": text,
                    "source": file,
                    "page": page_num + 1
                })

        print(f"Loaded {len(reader.pages)} pages from {file}")

    return docs


# usage
docs = load_pdfs("../dataset")
print(docs)

Loaded 7 pages from Data Engineering Resume Guide.pdf
Loaded 1 pages from SIH One Page Implementation.pdf
Loaded 1 pages from Trishansh.pdf
Loaded 7 pages from What is Docker.pdf
[{'text': 'Data Engineer Resume\nWhat Hiring Managers\nActually Look At\nvs. What You Think\nA side-by-side breakdown of every resume section\nso you stop making mistakes that get you rejected.\nby Darshil Parmar\nData Engineer | Educator | 100K+ Community\ndatavidhya.com\n7 Pages  |  6 Resume Sections\n1 / 7\n', 'source': 'Data Engineering Resume Guide.pdf', 'page': 1}, {'text': 'SECTION 1: Resume Headline & Summary\nYour headline is the first thing a recruiter sees. Most candidates waste it.\nHere\'s what they write vs. what actually gets attention:\nWhat Candidates Write\nWhat Hiring Managers Want\n"Aspiring Data Engineer | Passionate about Big\nData | Seeking Opportunities"\nVague, generic, says nothing about skills or\nexperience. Reads like every other resume.\n"Data Engineer | 3 yrs building ETL pipelin

## Chunking

In [2]:
def chunk_text(text, chunk_size=300, overlap=50):
    chunks = []
    start = 0

    while start < len(text):
        chunk = text[start:start + chunk_size]
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

## Embeddings 

In [3]:
from sentence_transformers import SentenceTransformer

class Embedder:
    def __init__(self):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")

    def encode(self, texts):
        return self.model.encode(texts, show_progress_bar=False)

d:\Bizfy Intern\Week - 1\Document_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## vector-store (FAISS)

In [4]:
import faiss
import numpy as np

class VectorStore:
    def __init__(self, dim):
        self.index = faiss.IndexFlatIP(dim)  # cosine similarity
        self.texts = []
        self.metadata = []

    def add(self, embeddings, texts, metadatas):
        embeddings = np.array(embeddings).astype("float32")
        faiss.normalize_L2(embeddings)

        self.index.add(embeddings)
        self.texts.extend(texts)
        self.metadata.extend(metadatas)

    def search(self, query_embedding, k=5):
        query_embedding = np.array([query_embedding]).astype("float32")
        faiss.normalize_L2(query_embedding)

        distances, indices = self.index.search(query_embedding, k)

        results = []
        for i in indices[0]:
            results.append({
                "text": self.texts[i],
                "metadata": self.metadata[i]
            })

        return results

## Pipeline

In [5]:
def build_pipeline(folder_path):
    docs = load_pdfs(folder_path)

    all_chunks = []
    metadatas = []

    for doc in docs:
        chunks = chunk_text(doc["text"])

        for chunk in chunks:
            all_chunks.append(chunk)
            metadatas.append({
                "source": doc["source"],
                "page": doc["page"]
            })

    if len(all_chunks) == 0:
        print("No text found in PDFs.")
        return None, None

    embedder = Embedder()
    embeddings = embedder.encode(all_chunks)

    dim = embeddings.shape[1]
    store = VectorStore(dim)
    store.add(embeddings, all_chunks, metadatas)

    return store, embedder


## Query System 

In [6]:
def query_system(store, embedder, query):
    query_embedding = embedder.encode([query])[0]

    results = store.search(query_embedding, k=5)

    print("\nTop Results:\n")
    for r in results:
        print("Source:", r["metadata"]["source"])
        print("Page:", r["metadata"]["page"])     
        print("Text:", r["text"][:200])
        print("-" * 50)

## main code 

In [8]:
if __name__ == "__main__":
    store, embedder = build_pipeline("../dataset")

    if store is None:
        exit()

    while True:
        q = input("\nEnter query: ")
        if q.lower() == "exit":
            break
        query_system(store, embedder, q)

Loaded 7 pages from Data Engineering Resume Guide.pdf
Loaded 1 pages from SIH One Page Implementation.pdf
Loaded 1 pages from Trishansh.pdf
Loaded 7 pages from What is Docker.pdf


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2387.58it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Top Results:

Source: Data Engineering Resume Guide.pdf
Page: 3
Text: e, Hadoop, Spark, Kafka, Airflow, Flink,
Docker, Kubernetes, Terraform, Jenkins, Git, Tableau,
Power BI, Excel...
Listing 25+ tools screams "I googled data engineering
skills."
Skills (grouped by func
--------------------------------------------------
Source: Data Engineering Resume Guide.pdf
Page: 6
Text: d Data Engineer (2024)
Only list industry-recognized certs. Two strong certs
beat fifteen course completions.
If you have no certs, skip this section entirely. It's
better to have nothing than to pad 
--------------------------------------------------
Source: Data Engineering Resume Guide.pdf
Page: 2
Text: onate about Big
Data | Seeking Opportunities"
Vague, generic, says nothing about skills or
experience. Reads like every other resume.
"Data Engineer | 3 yrs building ETL pipelines on
AWS | Python, SQL
--------------------------------------------------
Source: Data Engineering Resume Guide.pdf
Page: 6
Text: on:
B